# LatentSync 1.6 — GPU Colab POC

Runs **ByteDance LatentSync 1.6** (photorealistic latent-diffusion lip-sync) on a Colab GPU, then reports **timing** and **quality metrics** to compare with the other notebooks.

**Input:** LatentSync is *video → video*. Use the bundled demo video, or your own video (+ optional photo→static-video cell).

> Set **Runtime → Change runtime type → GPU** first.

## Step 0 — confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Step 1 — clone LatentSync + install deps

In [ ]:
%cd /content
!git clone https://github.com/bytedance/LatentSync.git
%cd /content/LatentSync
!apt-get -qq install -y ffmpeg libgl1 > /dev/null
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

## Step 2 — download LatentSync 1.6 checkpoints (a few GB)

In [ ]:
from huggingface_hub import snapshot_download
import os
os.makedirs('checkpoints', exist_ok=True)
snapshot_download(repo_id='ByteDance/LatentSync-1.6', local_dir='checkpoints',
    allow_patterns=['latentsync_unet.pt', 'whisper/tiny.pt', 'stable_syncnet.pt'])
!ls -lah checkpoints checkpoints/whisper

## Step 3 — choose inputs
**A (default):** bundled demo footage.  **B:** your own video+audio.  **C:** photo → static video (weaker).

In [ ]:
# Option A - bundled demo
!ls assets | head
VIDEO_PATH = 'assets/demo1_video.mp4'
AUDIO_PATH = 'assets/demo1_audio.wav'

In [ ]:
# Option B - upload your own (uncomment)
# from google.colab import files
# up = files.upload()
# names = list(up.keys())
# VIDEO_PATH = [n for n in names if n.lower().endswith(('.mp4','.mov','.avi'))][0]
# AUDIO_PATH = [n for n in names if n.lower().endswith(('.wav','.mp3','.m4a'))][0]

In [ ]:
# Option C - photo -> static video (uncomment; same face as other demos)
# from google.colab import files, subprocess
# up = files.upload()   # sample_face.png + sample_audio.wav
# dur = subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration',
#     '-of','default=nw=1:nk=1','sample_audio.wav']).decode().strip()
# !ffmpeg -y -loop 1 -i sample_face.png -t {dur} -r 25 -vf scale=512:512 -pix_fmt yuv420p static_face.mp4
# VIDEO_PATH = 'static_face.mp4'; AUDIO_PATH = 'sample_audio.wav'

## Step 4 — run LatentSync + timing
`inference_steps` (higher=sharper/slower) and `guidance_scale` (lip expressiveness) are the knobs.

In [ ]:
import time; _t = time.time()
OUT_PATH = '/content/latentsync_out.mp4'
!python -m scripts.inference \
  --unet_config_path 'configs/unet/stage2_512.yaml' \
  --inference_ckpt_path 'checkpoints/latentsync_unet.pt' \
  --inference_steps 20 --guidance_scale 1.5 \
  --video_path "$VIDEO_PATH" --audio_path "$AUDIO_PATH" --video_out_path "$OUT_PATH"
print(f'[TIME] LatentSync end-to-end: {time.time()-_t:.1f}s')
# If flags differ in your cloned version:  !python -m scripts.inference --help

## Step 5 — quality metrics

In [ ]:
# Quality metrics (reference-free): CSIM identity + mouth/face sharpness.
# Same definitions as the local wav2lip/evaluate.py, so numbers are comparable
# across notebooks. Needs OpenCV YuNet + SFace (downloaded here if missing).
import urllib.request, os, cv2, numpy as np
_M = {
 "yunet.onnx": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
 "sface.onnx": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
}
for _n, _u in _M.items():
    if not os.path.exists(_n):
        urllib.request.urlretrieve(_u, _n)
_det = cv2.FaceDetectorYN.create("yunet.onnx", "", (320, 320), 0.6, 0.3, 5000)
_rec = cv2.FaceRecognizerSF.create("sface.onnx", "")

def _big(f):
    h, w = f.shape[:2]; _det.setInputSize((w, h)); _, fc = _det.detect(f)
    return None if fc is None or len(fc) == 0 else max(fc, key=lambda z: z[-1])

def _lap(g):
    return float(cv2.Laplacian(g, cv2.CV_64F).var())

def _mouth(fr, face):
    lm = face[4:14].reshape(5, 2); (rx, ry), (lx, ly) = lm[3], lm[4]
    cx, cy = int((rx + lx) / 2), int((ry + ly) / 2)
    mw = int(abs(lx - rx) * 1.6) or 40; mh = int(mw * 0.7)
    c = fr[max(0, cy - mh // 2):cy + mh // 2, max(0, cx - mw // 2):cx + mw // 2]
    return 0.0 if c.size == 0 else _lap(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY))

def _first(path):
    if path.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp")):
        return cv2.imread(path)
    cap = cv2.VideoCapture(path); ok, fr = cap.read(); cap.release(); return fr

def metrics(video, source, every=5):
    src = _first(source); sf = _big(src)
    sfeat = _rec.feature(_rec.alignCrop(src, sf))
    cap = cv2.VideoCapture(video); cs, ms, fs = [], [], []; i = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if i % every == 0:
            f = _big(fr)
            if f is not None:
                cs.append(float(_rec.match(sfeat, _rec.feature(_rec.alignCrop(fr, f)),
                                           cv2.FaceRecognizerSF_FR_COSINE)))
                ms.append(_mouth(fr, f)); x, y, bw, bh = f[:4].astype(int)
                roi = fr[max(0, y):y + bh, max(0, x):x + bw]
                if roi.size:
                    fs.append(_lap(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)))
        i += 1
    cap.release()
    _m = lambda a: float(np.mean(a)) if a else 0.0
    print(f"CSIM(identity)={_m(cs):.3f}   mouth_sharp={_m(ms):.1f}   "
          f"face_sharp={_m(fs):.1f}   frames={len(cs)}   [CSIM>0.6=same person]")


In [ ]:
print('== LatentSync =='); metrics(OUT_PATH, VIDEO_PATH)

## Step 6 — preview + download

In [ ]:
from IPython.display import HTML
from base64 import b64encode
d = b64encode(open(OUT_PATH,'rb').read()).decode()
display(HTML(f'<video width=512 controls><source src="data:video/mp4;base64,{d}" type="video/mp4"></video>'))

In [ ]:
from google.colab import files
files.download(OUT_PATH)